In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score



# Импортируем необходимые библиотеки для работы с данными и моделями машинного обучения
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score 
import xgboost as xgb
from catboost import CatBoostClassifier

# Указываем путь к данным на Kaggle
dataPath = "/kaggle/input/home-credit-credit-risk-model-stability/"

# Функция для установки типов данных в таблицах с использованием библиотеки Polars
def set_table_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    # Для каждого столбца в датафрейме проверяем последнюю букву его названия
    # и, в зависимости от неё, приводим столбец к определённому типу данных
    for col in df.columns:
        if col[-1] in ("P", "A"):
            df = df.with_columns(pl.col(col).cast(pl.Float64).alias(col))
    return df

# Функция для конвертации строковых столбцов в категории в датафрейме Pandas
def convert_strings(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:  
        if df[col].dtype.name in ['object', 'string']:
            df[col] = df[col].astype("string").astype('category')
            current_categories = df[col].cat.categories
            new_categories = current_categories.to_list() + ["Unknown"]
            new_dtype = pd.CategoricalDtype(categories=new_categories, ordered=True)
            df[col] = df[col].astype(new_dtype)
    return df

# Чтение и обработка обучающих данных с использованием Polars
train_basetable = pl.read_csv(dataPath + "csv_files/train/train_base.csv")
train_static = pl.concat(
    [
        pl.read_csv(dataPath + "csv_files/train/train_static_0_0.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/train/train_static_0_1.csv").pipe(set_table_dtypes),
    ],
    how="vertical_relaxed",
)
# Аналогично обрабатываем другие части обучающего набора данных
train_static_cb = pl.read_csv(dataPath + "csv_files/train/train_static_cb_0.csv").pipe(set_table_dtypes)
train_person_1 = pl.read_csv(dataPath + "csv_files/train/train_person_1.csv").pipe(set_table_dtypes) 
train_credit_bureau_b_2 = pl.read_csv(dataPath + "csv_files/train/train_credit_bureau_b_2.csv").pipe(set_table_dtypes) 

# Чтение и обработка тестовых данных, аналогично обучающим
test_basetable = pl.read_csv(dataPath + "csv_files/test/test_base.csv")
test_static = pl.concat(
    [
        pl.read_csv(dataPath + "csv_files/test/test_static_0_0.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/test/test_static_0_1.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/test/test_static_0_2.csv").pipe(set_table_dtypes),
    ],
    how="vertical_relaxed",
)
test_static_cb = pl.read_csv(dataPath + "csv_files/test/test_static_cb_0.csv").pipe(set_table_dtypes)
test_person_1 = pl.read_csv(dataPath + "csv_files/test/test_person_1.csv").pipe(set_table_dtypes) 
test_credit_bureau_b_2 = pl.read_csv(dataPath + "csv_files/test/test_credit_bureau_b_2.csv").pipe(set_table_dtypes) 

# Создание агрегированных признаков для обучающего набора данных
train_person_1_feats_1 = train_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

train_person_1_feats_2 = train_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

train_credit_bureau_b_2_feats = train_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)

# Выбор столбцов определённого типа из статических данных
selected_static_cols = [col for col in train_static.columns if col[-1] in ("A", "M")]
print(selected_static_cols)

selected_static_cb_cols = [col for col in train_static_cb.columns if col[-1] in ("A", "M")]
print(selected_static_cb_cols)

# Объединение всех таблиц в одну для обучения модели
data = train_basetable.join(
    train_static.select(["case_id"]+selected_static_cols), how="left", on="case_id"
).join(
    train_static_cb.select(["case_id"]+selected_static_cb_cols), how="left", on="case_id"
).join(
    train_person_1_feats_1, how="left", on="case_id"
).join(
    train_person_1_feats_2, how="left", on="case_id"
).join(
    train_credit_bureau_b_2_feats, how="left", on="case_id"
)


test_person_1_feats_1 = train_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

test_person_1_feats_2 = train_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

test_credit_bureau_b_2_feats = train_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)

data_submission = test_basetable.join(
    test_static.select(["case_id"]+selected_static_cols), how="left", on="case_id"
).join(
    test_static_cb.select(["case_id"]+selected_static_cb_cols), how="left", on="case_id"
).join(
    test_person_1_feats_1, how="left", on="case_id"
).join(
    test_person_1_feats_2, how="left", on="case_id"
).join(
    test_credit_bureau_b_2_feats, how="left", on="case_id"
)

case_ids = data["case_id"].unique().shuffle(seed=1)
case_ids_train, case_ids_test = train_test_split(case_ids, train_size=0.6, random_state=1)
case_ids_valid, case_ids_test = train_test_split(case_ids_test, train_size=0.5, random_state=1)

cols_pred = []
for col in data.columns:
    if col[-1].isupper() and col[:-1].islower():
        cols_pred.append(col)

print(cols_pred)


# Определение функции для конвертации данных из Polars в Pandas для тренировочного, валидационного и тестового наборов
def from_polars_to_pandas(case_ids: pl.DataFrame) -> pl.DataFrame:
    # Фильтрация данных по уникальным идентификаторам и выбор необходимых столбцов, конвертация в Pandas DataFrame
    return (
        data.filter(pl.col("case_id").is_in(case_ids))[["case_id", "WEEK_NUM", "target"]].to_pandas(),
        data.filter(pl.col("case_id").is_in(case_ids))[cols_pred].to_pandas(),
        data.filter(pl.col("case_id").is_in(case_ids))["target"].to_pandas()
    )

# Получение данных для обучения, валидации и тестирования
base_train, X_train, y_train = from_polars_to_pandas(case_ids_train)
base_valid, X_valid, y_valid = from_polars_to_pandas(case_ids_valid)
base_test, X_test, y_test = from_polars_to_pandas(case_ids_test)

# Применение функции конвертации строковых столбцов для каждого DataFrame
for df in [X_train, X_valid, X_test]:
    df = convert_strings(df)

# Вывод размеров обучающего, валидационного и тестового наборов данных
print(f"Train: {X_train.shape}")
print(f"Valid: {X_valid.shape}")
print(f"Test: {X_test.shape}")

# Вывод числа деревьев для модели
print(f" n_estimators: 3000")

# Параметры для модели LightGBM
params = {
    "boosting_type": "gbdt",  # Тип алгоритма
    "objective": "binary",  # Задача бинарной классификации
    "metric": "auc",  # Метрика качества
    "max_depth": 16,  # Максимальная глубина дерева
    "num_leaves": 50,  # Количество листьев в дереве
    "learning_rate": 0.005,  # Скорость обучения
    "feature_fraction": 0.8,  # Доля признаков для обучения каждого дерева
    "bagging_fraction": 0.8,  # Доля объектов для обучения каждого дерева
    "bagging_freq": 5,  # Частота использования bagging
    "n_estimators": 3000,  # Количество деревьев 3000
    "verbose": -1,  # Уровень логирования
    'device':'gpu'  # Использование GPU
}

# Настройка и использование StratifiedKFold для валидации
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

lgb_models = []  # Список для сохранения моделей

# Обучение модели LightGBM с использованием кросс-валидации
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    lgb_train = lgb.Dataset(X_train_fold, label=y_train_fold)
    lgb_valid = lgb.Dataset(X_valid_fold, label=y_valid_fold, reference=lgb_train)

    gbm = lgb.train(params, lgb_train, valid_sets=[lgb_valid],callbacks=[lgb.log_evaluation(50), lgb.early_stopping(10)])
    lgb_models.append(gbm)

# Обучение модели XGBoost
print(f" XGBClassifier")
xgb_model = xgb.XGBClassifier(
    device="cuda",  # Использование GPU
    objective='binary:logistic',  # Задача бинарной классификации
    tree_method="hist",  # Метод построения деревьев
    enable_categorical=True,  # Включение поддержки категориальных признаков
    eval_metric='auc',  # Метрика качества
    subsample=1,
    colsample_bytree=1,
    min_child_weight=1,
    max_depth=20,  # Максимальная глубина дерева
    n_estimators=1200,  # Количество деревьев  1200
    random_state=42,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    early_stopping_rounds=100,
    verbose=True,
)

# Обработка категориальных признаков для модели CatBoost
cat_features = [col for col in X_train.columns if X_train[col].dtype.name == 'category' or X_train[col].dtype.name == 'object']
for col in cat_features:
    X_train[col] = X_train[col].cat.add_categories('Missing').fillna('Missing')
    X_valid[col] = X_valid[col].cat.add_categories('Missing').fillna('Missing')

# Обучение модели CatBoost
print(f" CatBoostClassifier")
cat_model = CatBoostClassifier(
    iterations=2000, #1200                
    depth=12,                        
    learning_rate=0.05,               
    eval_metric='AUC',               
    random_seed=42,                  
    bootstrap_type='Bayesian',       
    bagging_temperature=1,           
    od_type='Iter',                  
    od_wait=50,
    task_type='GPU'
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features,  
    use_best_model=True,
    verbose=True
)

# Подготовка данных для финального предсказания
X_submission = data_submission[cols_pred].to_pandas()
X_submission = convert_strings(X_submission)
X_submission_processed = X_submission.copy()
categorical_cols = X_train.select_dtypes(include=['category']).columns

for col in categorical_cols:
    train_categories = set(X_train[col].cat.categories)
    submission_categories = set(X_submission[col].cat.categories)
    new_categories = submission_categories - train_categories
    X_submission.loc[X_submission[col].isin(new_categories), col] = "Unknown"
    new_dtype = pd.CategoricalDtype(categories=train_categories, ordered=True)
    X_train[col] = X_train[col].astype(new_dtype)
    X_submission[col] = X_submission[col].astype(new_dtype)

for col in cat_features:
    X_submission_processed[col] = X_submission_processed[col].cat.add_categories('Missing').fillna('Missing')

# Функция для усреднения предсказаний от разных моделей
def predict_with_fold_models(models, X):
    predictions = [model.predict(X, num_iteration=model.best_iteration) for model in models]
    return np.mean(predictions, axis=0)

# Выполнение предсказаний для каждой модели
lgb_pred = predict_with_fold_models(lgb_models, X_submission) 
xgb_pred = xgb_model.predict(X_submission)
cat_pred = cat_model.predict(X_submission_processed)

# Усреднение предсказаний с весами
weight_lgb = 0.6
weight_xgb = 0.2 
weight_cat = 0.2
assert weight_lgb + weight_xgb + weight_cat == 1, "The sum of weights must be 1."

y_submission_pred = (weight_lgb * lgb_pred) + (weight_xgb * xgb_pred) + (weight_cat * cat_pred)

# Создание и сохранение файла с предсказаниями
submission = pd.DataFrame({
    "case_id": data_submission["case_id"].to_numpy(),
    "score": y_submission_pred
}).set_index('case_id')
submission.to_csv("./submission.csv")

submission